# Topic: Grounded or Hallucinated? A Small RAG System that Knows When to Say "I Don't Know"
---
## General Details

**Course:** Deep Learning [BMEVITMMA19]  
**Program:** Data Science and Artificial Intelligence, MSc  
**Department:** Department of Telecommunications and Artificial Intelligence  
**Instructor:**  Mohammed Salah Hamza Al-Radhi   
**Academic year:** 2026/27/1  
**Team members:** Ruvejda Gjyrevci [FQL6EH],  Rahma Nouri [EZDR82], Arlind Bajraktari [D4104N]

---

## Project Overview

Retrieval-Augmented Generation (RAG) allows a language model to answer questions using information retrieved from external documents rather than relying only on its internal knowledge.

This project uses the MultiHop-RAG dataset, which contains 2,556 questions over 609 news articles. Some questions require combining information from multiple documents, while approximately 12% of the questions are intentionally unanswerable from the available articles.

The main goal is to build and evaluate a RAG system that can retrieve relevant evidence, answer multi-hop questions, and recognize when sufficient evidence is unavailable.

---


## Milestone 1 – Reproduce

The goal of Milestone 1 is to reproduce the retrieval results of the MultiHop-RAG repository.

The main tasks are:

1. Clone the MultiHop-RAG repository.
2. Install the required dependencies and the specified `llama-index` version.
3. Download `corpus.json` and `MultiHopRAG.json`.
4. Run dense retrieval using `bge-large-en-v1.5`.
5. Use a chunk size of 256 tokens and record the chunk-overlap setting.
6. Evaluate the retrieval results.
7. Run BM25 as a keyword-based baseline.

### Target Results

- Hits@10: **0.6718**
- Hits@4: **0.5221**
- MRR@10: **0.4298**
- Hits@10 with `bge-reranker-large`: **0.7183**

## T1 – Clone the MultiHop-RAG Repository

The first step is to download the official MultiHop-RAG repository from GitHub into the Google Colab environment.

This repository contains the code required to reproduce the retrieval experiments, including the dense retrieval script, the evaluation script, and the BM25 baseline.

After cloning the repository, we move into the project directory so that all following commands are executed from the correct location.

Assigned to: Ruvejda Gjyrevci

In [1]:
# T1: Clone the repository (skipped automatically if it is already there)
%cd /content
![ -d MultiHop-RAG ] || git clone https://github.com/yixuantt/MultiHop-RAG.git
%cd /content/MultiHop-RAG

/content
Cloning into 'MultiHop-RAG'...
remote: Enumerating objects: 164, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 164 (delta 27), reused 83 (delta 16), pack-reused 62 (from 1)
Receiving objects: 100% (164/164), 8.04 MiB | 10.50 MiB/s, done.
Resolving deltas: 100% (52/52), done.
/content/MultiHop-RAG


### T1.1 – Save all results to Google Drive (crash protection)

Colab deletes every file when the runtime stops. To avoid losing hours of computation, the folders `dataset/`, `output/` and `storage/` are stored on Google Drive (linked into the repository).
If the runtime stops, reconnect and run **all cells again from the top**: the expensive steps find their saved results and are skipped, and the LLM steps continue from where they stopped.

In [2]:
# T1.1: Store dataset/, output/ and storage/ on Google Drive so nothing is lost if Colab stops
from google.colab import drive
drive.mount("/content/drive")

import os, shutil
SAVE_DIR = "/content/drive/MyDrive/MultiHopRAG_run"
os.chdir("/content/MultiHop-RAG")
for d in ["dataset", "output", "storage"]:
    target = os.path.join(SAVE_DIR, d)
    os.makedirs(target, exist_ok=True)
    if os.path.islink(d):
        continue
    if os.path.isdir(d):                      # move anything already created locally to Drive
        for name in os.listdir(d):
            if not os.path.exists(os.path.join(target, name)):
                shutil.move(os.path.join(d, name), target)
        shutil.rmtree(d)
    os.symlink(target, d)

for d in ["dataset", "output", "storage"]:
    print(f"{d:8s} -> {os.path.realpath(d)}   ({len(os.listdir(d))} items already saved)")

Mounted at /content/drive
dataset  -> /content/drive/MyDrive/MultiHopRAG_run/dataset   (2 items already saved)
output   -> /content/drive/MyDrive/MultiHopRAG_run/output   (8 items already saved)
storage  -> /content/drive/MyDrive/MultiHopRAG_run/storage   (1 items already saved)


## T2 – Inspect the README and Project Requirements

The README contains the official instructions for setting up and running the MultiHop-RAG repository.

For Milestone 1, the README is used to identify the required dependency versions and the main scripts needed for retrieval and evaluation.

Assigned to: Ruvejda Gjyrevci

In [3]:
!cat README.md

# 💡 MultiHop-RAG
A Dataset for **Evaluating Retrieval-Augmented Generation Across Documents**  

   
## 🚀 Overview
**MultiHop-RAG**: a QA dataset to evaluate retrieval and reasoning across documents with metadata in the RAG pipelines. It contains 2556 queries, with evidence for each query distributed across 2 to 4 documents. The queries also involve document metadata, reflecting complex scenarios commonly found in real-world RAG applications.  

📄 Paper Link **(Accepted by COLM 2024)**: [MultiHop-RAG: Benchmarking Retrieval-Augmented Generation for Multi-Hop Queries](https://arxiv.org/pdf/2401.15391.pdf)  
🤗 [Hugging Face dataloader](https://huggingface.co/datasets/yixuantt/MultiHopRAG)

![rag.png](resource/rag.png)

## Simple Use Case

**1. For Retrieval**

Please try '**simple_retrieval.py**,' a sample use case demonstrating retrieval using this dataset. 
```
pip install llama-index==0.9.40
```
```shell
# test simple retrieval and save results
python simple_retrieval.py --retriever B

## T3 – Install the Required Dependency

The MultiHop-RAG repository requires a specific version of the LlamaIndex library.

According to the repository README, `llama-index==0.9.40` must be installed to reproduce the original setup correctly.

This step prepares the Colab environment so that the repository scripts can run with the expected library version.

Assigned to: Ruvejda Gjyrevci

In [4]:
# Install the exact LlamaIndex version specified in the repository README
!pip install -q llama-index==0.9.40

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires tenacity<10,>=9, but you have tenacity 8.5.0 which is incompatible.


In [5]:
# Import LlamaIndex to confirm that the required version works
import llama_index

# Print a confirmation message if the import succeeds
print("LlamaIndex imported successfully.")

/usr/local/lib/python3.13/dist-packages/llama_index/download/module.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


LlamaIndex imported successfully.


## T4 – Create the Dataset Folder

The MultiHop-RAG repository expects the dataset files to be stored inside a folder named `dataset`.

This folder will contain the two files required for Milestone 1:

- `corpus.json`
- `MultiHopRAG.json`

Creating the folder first keeps the project structure consistent with the repository requirements.

Assigned to: Ruvejda Gjyrevci

In [6]:
# Create the dataset folder inside the current MultiHop-RAG repository
!mkdir -p dataset

# Display the current repository contents to confirm the folder was created
!ls

agentic_rag	     qa_llama.py	    run_agentic_rag.py
bm25_retrieval.py    README.md		    run_hybrid_retrieval.py
dataset		     requirement.txt	    simple_retrieval.py
hybrid_retriever.py  rerank_result.py	    storage
output		     resource		    tests
pipeline	     retrieval_evaluate.py  util.py
qa_evaluate.py	     rrf_fusion.py


## T5 – Download the Required Dataset Files

The MultiHop-RAG dataset is hosted on Hugging Face.

For Milestone 1, two files are required:

- `corpus.json` – contains the news article corpus used for retrieval.
- `MultiHopRAG.json` – contains the questions and related evaluation information.

Both files are downloaded into the `dataset` folder created in the previous task.

Assigned to: Ruvejda Gjyrevci

In [7]:
# Download the corpus file from the MultiHopRAG dataset on Hugging Face
!wget -O dataset/corpus.json \
https://huggingface.co/datasets/yixuantt/MultiHopRAG/resolve/main/corpus.json

# Download the question and evaluation file from the same dataset
!wget -O dataset/MultiHopRAG.json \
https://huggingface.co/datasets/yixuantt/MultiHopRAG/resolve/main/MultiHopRAG.json

--2026-11-18 09:16:58--  https://huggingface.co/datasets/yixuantt/MultiHopRAG/resolve/main/corpus.json
Resolving huggingface.co (huggingface.co)... 13.35.202.121, 13.35.202.34, 13.35.202.97, ...
Connecting to huggingface.co (huggingface.co)|13.35.202.121|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: /api/resolve-cache/datasets/yixuantt/MultiHopRAG/71ac0d0bd1f951d2d6b70311f7d2ae404e1ffa82/corpus.json?%2Fdatasets%2Fyixuantt%2FMultiHopRAG%2Fresolve%2Fmain%2Fcorpus.json=&etag=%22bb98345ef3921312aad05fd117b5a3e39888de2c%22 [following]
--2026-11-18 09:16:58--  https://huggingface.co/api/resolve-cache/datasets/yixuantt/MultiHopRAG/71ac0d0bd1f951d2d6b70311f7d2ae404e1ffa82/corpus.json?%2Fdatasets%2Fyixuantt%2FMultiHopRAG%2Fresolve%2Fmain%2Fcorpus.json=&etag=%22bb98345ef3921312aad05fd117b5a3e39888de2c%22
Reusing existing connection to huggingface.co:443.
HTTP request sent, awaiting response... 200 OK
Length: 6785567 (6.5M) [text/plain]
Saving to: ‘da

In [8]:
# Show the downloaded dataset files and their sizes
!ls -lh dataset

lrwxrwxrwx 1 root root 46 Nov 18 09:16 dataset -> /content/drive/MyDrive/MultiHopRAG_run/dataset


## T6 – Verify the Dataset Files

The downloaded dataset files are checked to confirm that they are stored in the correct location and can be loaded successfully.

This verification ensures that the dataset setup is complete before the retrieval experiments are started.

Assigned to: Ruvejda Gjyrevci

In [9]:
# Import the JSON library
import json

# Load the corpus file
with open("dataset/corpus.json", "r", encoding="utf-8") as f:
    corpus = json.load(f)

# Load the MultiHop-RAG question file
with open("dataset/MultiHopRAG.json", "r", encoding="utf-8") as f:
    questions = json.load(f)

# Print the number of records in each dataset file
print("Number of corpus records:", len(corpus))
print("Number of questions:", len(questions))

Number of corpus records: 609
Number of questions: 2556


## T7 – Configure Dense Retrieval

Dense retrieval uses embeddings to represent text as numerical vectors and retrieve passages that are semantically relevant to a query.

For this experiment, we selected the `BAAI/bge-large-en-v1.5` embedding model and configured the chunk size to 256 tokens. The chunk overlap was set to 25 tokens, allowing adjacent chunks to share some text and preserve context across chunk boundaries. We configured the retriever to return the top 10 passages for each query.

Assigned to: Rahma Nouri

In [10]:
# T7: Dense retrieval configuration

MODEL_NAME = "BAAI/bge-large-en-v1.5"
CHUNK_SIZE = 256
CHUNK_OVERLAP = 25
TOP_K = 10

print("Dense retrieval configuration")
print("Embedding model:", MODEL_NAME)
print("Chunk size:", CHUNK_SIZE, "tokens")
print("Chunk overlap:", CHUNK_OVERLAP, "tokens")
print("Top-k retrieval:", TOP_K)

assert 0 <= CHUNK_OVERLAP < CHUNK_SIZE

Dense retrieval configuration
Embedding model: BAAI/bge-large-en-v1.5
Chunk size: 256 tokens
Chunk overlap: 25 tokens
Top-k retrieval: 10


## T8 – Run Dense Retrieval

We ran the dense retrieval pipeline using `BAAI/bge-large-en-v1.5`, a chunk size of 256 tokens, a chunk overlap of 25 tokens, and top-k = 10.

The pipeline loaded the 609 articles in `corpus.json`, split them into 8,034 chunks, generated embeddings, and built a vector index. It then retrieved the top 10 passages for each of the 2,556 records in `MultiHopRAG.json`.

The retrieval results were saved to `output/bge-large-en-v1.5_retrieval_test.json` for evaluation.

Assigned to: Rahma Nouri

In [13]:
# T8: Run dense retrieval (skipped if the result is already saved on Drive)

%cd /content/MultiHop-RAG
import os

if os.path.exists("output/bge-large-en-v1.5_retrieval_test.json") and os.path.isdir("storage/bge-large-en-v1.5_chunk256"):
    print("Saved dense index and results found on Drive - skipping T8 (already computed).")
else:
    !python simple_retrieval.py \
        --retriever BAAI/bge-large-en-v1.5 \
        --chunk_size 256 \
        --chunk_overlap 25 \
        --topk 10 \
        --rebuild_index

/content/MultiHop-RAG
/usr/local/lib/python3.13/dist-packages/llama_index/download/module.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
persist_dir: storage/bge-large-en-v1.5_chunk256
save_file: output/bge-large-en-v1.5_retrieval_test.json
Loading weights: 100% 391/391 [00:00<00:00, 4486.62it/s]
rebuild_index=True, removing existing index: storage/bge-large-en-v1.5_chunk256
No persisted index found. Building a new index...
Total documents: 609
Total nodes/chunks: 8034
LLM sees:
 [Excerpt from document]
title: 200+ of the best deals from Amazon's Cyber Monday sale
published_at: 2023-11-27T08:45:59+00:00
source: Mashable
Excerpt:
-----
Table of Contents Table of Contents Echo, Fire TV, and Kindle deals Apple deals TV deals Laptop deals Headphone and earbud deals Tab

## T9 – Verify Dense Retrieval Output

After running the dense retrieval pipeline, we verified that the output file was generated successfully. The file contains retrieval results for all 2,556 query records, with the top 10 passages stored for each query.

The results were saved to `output/bge-large-en-v1.5_retrieval_test.json` and are ready for the separate evaluation step.

The experiment used `BAAI/bge-large-en-v1.5`, a chunk size of 256 tokens, and a chunk overlap of 25 tokens.

Assigned to: Rahma Nouri

In [14]:
# T9: Verify the dense retrieval output

import json
import os

output_file = "output/bge-large-en-v1.5_retrieval_test.json"

assert os.path.exists(output_file), "Retrieval output file not found."

with open(output_file, "r", encoding="utf-8") as f:
    results = json.load(f)

print("Output file exists:", os.path.exists(output_file))
print("File size:", f"{os.path.getsize(output_file):,}", "bytes")
print("Number of query records:", len(results))

if results:
    print("Retrieved passages for first query:",
          len(results[0]["retrieval_list"]))

assert len(results) == 2556
assert len(results[0]["retrieval_list"]) == 10

print("Dense retrieval output verified successfully.")

Output file exists: True
File size: 32,353,023 bytes
Number of query records: 2556
Retrieved passages for first query: 10
Dense retrieval output verified successfully.


## T10 – Evaluate Dense Retrieval

We evaluated the dense retrieval results from T9 with the repository’s evaluation script, retrieval_evaluate.py, using three metrics: Hits@10, Hits@4 and MRR@10. Hits@k is the fraction of queries where at least one gold evidence passage appears in the top k retrieved chunks. MRR@10 averages 1/rank of the first relevant chunk in the top 10, scoring 0 when none is found. A retrieved chunk counts as relevant when it contains a gold evidence fact. The 301 null queries have no gold evidence, so the script excludes them, which leaves 2,255 answerable queries. The script also reports MAP@10, which we include for completeness. We compare our results with the targets reported in the MultiHop-RAG paper: Hits@10 = 0.6718, Hits@4 = 0.5221 and MRR@10 = 0.4298.

Assigned to: Arlind Bajraktari

In [15]:
# T10: Evaluate dense retrieval with the repository's evaluation script

!python retrieval_evaluate.py --file output/bge-large-en-v1.5_retrieval_test.json

# Compare with the paper targets
import json
from retrieval_evaluate import calculate_metrics

TARGETS = {"Hits@10": 0.6718, "Hits@4": 0.5221, "MRR@10": 0.4298}

def evaluate(path):
    with open(path, "r", encoding="utf-8") as f:
        data = [d for d in json.load(f) if d["question_type"] != "null_query"]
    retrieved = [[m["text"] for m in d["retrieval_list"]] for d in data]
    gold = [[m["fact"] for m in d["gold_list"]] for d in data]
    return calculate_metrics(retrieved, gold)

dense_metrics = evaluate("output/bge-large-en-v1.5_retrieval_test.json")

print(f"\n{'Metric':10s} {'Ours':>8s} {'Target':>8s} {'Diff':>8s}")
for k in TARGETS:
    print(f"{k:10s} {dense_metrics[k]:8.4f} {TARGETS[k]:8.4f} {dense_metrics[k] - TARGETS[k]:+8.4f}")

Evaluate file: output/bge-large-en-v1.5_retrieval_test.json
For file: output/bge-large-en-v1.5_retrieval_test.json
Evaluating 2255 valid queries.
Hits@10: 0.7996
Hits@4: 0.6058
MAP@10: 0.2057
MRR@10: 0.4372
--------------------

Metric         Ours   Target     Diff
Hits@10      0.7996   0.6718  +0.1278
Hits@4       0.6058   0.5221  +0.0837
MRR@10       0.4372   0.4298  +0.0074


## T11 – BM25 Keyword Baseline

As a keyword-based baseline, we ran the repository’s BM25 script, bm25_retrieval.py, which uses Okapi BM25 from the rank_bm25 library. It splits the corpus with the same settings as the dense run (256 tokens, overlap 25), giving 8,034 chunks. Each chunk includes the article’s title, source and publication date, as in the dense run. Because both retrievers search the same chunks, any difference in the scores comes from the retrieval method itself. For each of the 2,556 queries we retrieved the top 10 chunks and saved the results to output/bm25_retrieval_test.json. We then evaluated them with retrieval_evaluate.py, using the same metrics as in T10, and compared BM25 against dense retrieval.

Assigned to: Arlind Bajraktari

In [16]:
# T11: BM25 keyword baseline with the repository's BM25 script

!pip install -q rank_bm25

import os
if os.path.exists("output/bm25_retrieval_test.json"):
    print("Saved BM25 results found on Drive - skipping the BM25 run.")
else:
    !python bm25_retrieval.py \
        --chunk_size 256 \
        --chunk_overlap 25 \
        --topk 10 \
        --output output/bm25_retrieval_test.json

!python retrieval_evaluate.py --file output/bm25_retrieval_test.json

bm25_metrics = evaluate("output/bm25_retrieval_test.json")

print(f"\n{'Retriever':28s} {'Hits@10':>8s} {'Hits@4':>8s} {'MRR@10':>8s}")
print(f"{'Paper target (dense)':28s} {TARGETS['Hits@10']:8.4f} {TARGETS['Hits@4']:8.4f} {TARGETS['MRR@10']:8.4f}")
for name, m in [("Dense bge-large-en-v1.5", dense_metrics), ("BM25", bm25_metrics)]:
    print(f"{name:28s} {m['Hits@10']:8.4f} {m['Hits@4']:8.4f} {m['MRR@10']:8.4f}")

Saved BM25 results found on Drive - skipping the BM25 run.
Evaluate file: output/bm25_retrieval_test.json
For file: output/bm25_retrieval_test.json
Evaluating 2255 valid queries.
Hits@10: 0.8683
Hits@4: 0.7126
MAP@10: 0.2909
MRR@10: 0.5747
--------------------

Retriever                     Hits@10   Hits@4   MRR@10
Paper target (dense)           0.6718   0.5221   0.4298
Dense bge-large-en-v1.5        0.7996   0.6058   0.4372
BM25                           0.8683   0.7126   0.5747


## Milestone 2 – Improve

The goal of Milestone 2 is to improve the baseline RAG system developed in Milestone 1 by strengthening retrieval, improving the handling of complex multi-hop questions, and adding answer generation.

The main tasks are:

1. Combine BM25 and dense retrieval using the repository's hybrid retrieval approach.
2. Add a re-ranker to improve the ranking of retrieved evidence.
3. Split complex multi-hop questions into simpler sub-questions and retrieve evidence for them separately.
4. Generate answers using a small open language model loaded in 4-bit precision.

### Target Results

No specific numerical target values are provided for Milestone 2.

The expected outcome is to measure whether the added components improve the baseline system from Milestone 1, particularly in terms of retrieval quality and answer generation.

##T12 – Hybrid BM25 + Dense Retrieval

To improve retrieval performance, BM25 keyword retrieval and dense retrieval are combined into a hybrid retrieval pipeline. BM25 is effective for exact keyword and entity matching, while dense retrieval is better at capturing semantic similarity between the question and the document chunks. By combining both approaches, the system can benefit from the strengths of lexical and semantic retrieval at the same time.

The repository’s hybrid retrieval implementation should be used with the same MultiHop-RAG corpus and, where applicable, the same retrieval settings used during Milestone 1. The hybrid results should be saved and evaluated using the same retrieval metrics so that they can be compared fairly with the individual BM25 and dense retrieval baselines.

Assigned to: Arlind Bajraktari

### 12.1 – Run Hybrid RRF Fusion

In [17]:
# T12.1: Fuse the dense (T8) and BM25 (T11) top-10 lists with Reciprocal Rank Fusion (RRF)

!python rrf_fusion.py \
    --dense output/bge-large-en-v1.5_retrieval_test.json \
    --bm25 output/bm25_retrieval_test.json \
    --output output/hybrid_rrf_retrieval_test.json \
    --topk 10 \
    --rrf_k 60 \
    --dense_weight 1.0 \
    --bm25_weight 1.0

Saved RRF results to: output/hybrid_rrf_retrieval_test.json


### 12.2 – Verify the Hybrid Output

In [18]:
# T12: Verify the hybrid retrieval output

import json
import os

hybrid_file = "output/hybrid_rrf_retrieval_test.json"

# Check that the file exists
assert os.path.exists(hybrid_file), "Hybrid retrieval output file not found."

# Load the saved hybrid results
with open(hybrid_file, "r", encoding="utf-8") as f:
    hybrid_results = json.load(f)

# Verify the number of records and retrieved passages
print("Output file exists:", os.path.exists(hybrid_file))
print("Number of query records:", len(hybrid_results))

if hybrid_results:
    print(
        "Retrieved passages for first query:",
        len(hybrid_results[0]["retrieval_list"])
    )

assert len(hybrid_results) == 2556
assert len(hybrid_results[0]["retrieval_list"]) == 10

print("Hybrid retrieval output verified successfully.")

Output file exists: True
Number of query records: 2556
Retrieved passages for first query: 10
Hybrid retrieval output verified successfully.


### T12.3 – Evaluate Hybrid Retrieval

In [19]:
# Evaluate the hybrid RRF retrieval results

!python retrieval_evaluate.py \
    --file output/hybrid_rrf_retrieval_test.json

Evaluate file: output/hybrid_rrf_retrieval_test.json
For file: output/hybrid_rrf_retrieval_test.json
Evaluating 2255 valid queries.
Hits@10: 0.8718
Hits@4: 0.7064
MAP@10: 0.2784
MRR@10: 0.5618
--------------------


### 12.4 – Compare Dense, BM25 and Hybrid retrieval

In [20]:
# T12: Compare Dense, BM25 and Hybrid retrieval

hybrid_metrics = evaluate("output/hybrid_rrf_retrieval_test.json")

print(f"\n{'Retriever':28s} {'Hits@10':>8s} {'Hits@4':>8s} {'MRR@10':>8s}")

for name, metrics in [
    ("Dense bge-large-en-v1.5", dense_metrics),
    ("BM25", bm25_metrics),
    ("Hybrid Dense + BM25", hybrid_metrics),
]:
    print(
        f"{name:28s} "
        f"{metrics['Hits@10']:8.4f} "
        f"{metrics['Hits@4']:8.4f} "
        f"{metrics['MRR@10']:8.4f}"
    )


Retriever                     Hits@10   Hits@4   MRR@10
Dense bge-large-en-v1.5        0.7996   0.6058   0.4372
BM25                           0.8683   0.7126   0.5747
Hybrid Dense + BM25            0.8718   0.7064   0.5618


## T12 – Result

The hybrid BM25 + Dense Retrieval approach achieved the highest Hits@10 score of **0.8718**, compared with **0.8683** for BM25 and **0.7996** for dense retrieval.

However, BM25 achieved the highest Hits@4 (**0.7126**) and MRR@10 (**0.5747**). The hybrid approach therefore slightly improved the probability of retrieving relevant evidence within the top 10 results, while BM25 alone ranked relevant evidence slightly higher near the top of the retrieval list.

This shows that combining dense and keyword-based retrieval improved overall retrieval coverage, but did not outperform BM25 across all evaluation metrics.

##T13 – Add `bge-reranker-large` and Measure Retrieval Improvement

A re-ranking stage is added after the initial retrieval step using the bge-reranker-large model. The retriever first produces a set of candidate passages, and the re-ranker then evaluates these candidates more carefully with respect to the original question.

The purpose of this task is to determine whether re-ranking improves the ordering of relevant evidence. The reranked output should be evaluated using the same retrieval metrics used in Milestone 1, such as Hits@10, Hits@4 and MRR@10. The results should then be compared with the corresponding retrieval results before re-ranking so that the improvement introduced by the re-ranker can be measured.

Assigned to: Ruvejda Gjyrevci

### 13.1 – Prepare Larger Candidate Sets

In [21]:
# T13: Prepare larger candidate pools for reranking (each step skipped if already saved on Drive)

%cd /content/MultiHop-RAG
import os

DENSE10 = "output/bge-large-en-v1.5_retrieval_test.json"
BACKUP  = "output/bge-large-en-v1.5_top10_backup.json"
DENSE20 = "output/bge-large-en-v1.5_top20_retrieval_test.json"
BM25_20 = "output/bm25_top20_retrieval_test.json"

# Dense retrieval: top-20 candidates. The persisted index from Milestone 1 is reused (no rebuild).
if os.path.exists(DENSE20):
    print("Saved dense top-20 found on Drive - skipping.")
else:
    # Preserve the original dense top-10 result (the script writes to the same file name)
    if not os.path.exists(BACKUP):
        !cp {DENSE10} {BACKUP}
    !python simple_retrieval.py \
        --retriever BAAI/bge-large-en-v1.5 \
        --chunk_size 256 \
        --chunk_overlap 25 \
        --topk 20
    !mv {DENSE10} {DENSE20}
    !cp {BACKUP} {DENSE10}

# BM25 retrieval: top-20 candidates
if os.path.exists(BM25_20):
    print("Saved BM25 top-20 found on Drive - skipping.")
else:
    !python bm25_retrieval.py \
        --chunk_size 256 \
        --chunk_overlap 25 \
        --topk 20 \
        --output {BM25_20}

/content/MultiHop-RAG
Saved dense top-20 found on Drive - skipping.
Saved BM25 top-20 found on Drive - skipping.


### 13.2 – Create Hybrid Top-20 Candidate Pool

In [23]:
# T13: Fuse the larger dense and BM25 candidate pools using RRF

!python rrf_fusion.py \
    --dense output/bge-large-en-v1.5_top20_retrieval_test.json \
    --bm25 output/bm25_top20_retrieval_test.json \
    --output output/hybrid_rrf_top20_candidates.json \
    --topk 20 \
    --rrf_k 60 \
    --dense_weight 1.0 \
    --bm25_weight 1.0

Saved RRF results to: output/hybrid_rrf_top20_candidates.json


### 13.3 – Verify the Candidate Pool

In [24]:
# Verify the hybrid top-20 candidate file

import json
import os

candidate_file = "output/hybrid_rrf_top20_candidates.json"

assert os.path.exists(candidate_file), "Hybrid candidate file not found."

with open(candidate_file, "r", encoding="utf-8") as f:
    hybrid_candidates = json.load(f)

print("Output file exists:", os.path.exists(candidate_file))
print("Number of query records:", len(hybrid_candidates))
print(
    "Candidates for first query:",
    len(hybrid_candidates[0]["retrieval_list"])
)

assert len(hybrid_candidates) == 2556
assert len(hybrid_candidates[0]["retrieval_list"]) == 20

print("Hybrid top-20 candidate pool verified successfully.")

Output file exists: True
Number of query records: 2556
Candidates for first query: 20
Hybrid top-20 candidate pool verified successfully.


### 13.4 – Create the Fair Pre-Reranking Baseline

In [25]:
# Create a top-10 baseline from the same hybrid top-20 candidate pool

pre_rerank_results = []

for item in hybrid_candidates:
    result = dict(item)
    result["retrieval_list"] = item["retrieval_list"][:10]
    pre_rerank_results.append(result)

pre_rerank_file = "output/hybrid_rrf_top20pool_top10_baseline.json"

with open(pre_rerank_file, "w", encoding="utf-8") as f:
    json.dump(pre_rerank_results, f, ensure_ascii=False, indent=2)

print("Saved pre-reranking baseline to:", pre_rerank_file)

Saved pre-reranking baseline to: output/hybrid_rrf_top20pool_top10_baseline.json


In [26]:
# Evaluate hybrid ranking before reranking

!python retrieval_evaluate.py \
    --file output/hybrid_rrf_top20pool_top10_baseline.json

Evaluate file: output/hybrid_rrf_top20pool_top10_baseline.json
For file: output/hybrid_rrf_top20pool_top10_baseline.json
Evaluating 2255 valid queries.
Hits@10: 0.8647
Hits@4: 0.7162
MAP@10: 0.2786
MRR@10: 0.5653
--------------------


### 13.5 – Install the BGE Reranker Dependency

In [27]:
# Install the FlagEmbedding package required by the BGE reranker

!pip install -q FlagEmbedding

In [28]:
# Record the installed FlagEmbedding version for reproducibility

import importlib.metadata

print(
    "FlagEmbedding version:",
    importlib.metadata.version("FlagEmbedding")
)

FlagEmbedding version: 1.4.2


### 13.6 – Apply `bge-reranker-large`

In [29]:
# T13: Re-rank the hybrid top-20 candidates with bge-reranker-large
# (skipped if the reranked file is already saved on Drive)

import os

if os.path.exists("output/hybrid_bge_reranker_top10.json"):
    print("Saved reranked results found on Drive - skipping the reranking run.")
else:
    # T13: Re-rank the hybrid top-20 candidates with bge-reranker-large

    import json
    from tqdm.auto import tqdm

    from llama_index.postprocessor import FlagEmbeddingReranker
    from llama_index.schema import TextNode, NodeWithScore, QueryBundle

    input_file = "output/hybrid_rrf_top20_candidates.json"
    output_file = "output/hybrid_bge_reranker_top10.json"

    # Load the hybrid candidate results
    with open(input_file, "r", encoding="utf-8") as f:
        candidate_data = json.load(f)

    # Load the required BGE reranker
    reranker = FlagEmbeddingReranker(
        model="BAAI/bge-reranker-large",
        top_n=10
    )

    reranked_results = []

    for item in tqdm(candidate_data, desc="Reranking queries"):

        # Convert the saved candidate passages into LlamaIndex nodes
        nodes = [
            NodeWithScore(
                node=TextNode(text=candidate["text"]),
                score=float(candidate.get("score", 0.0))
            )
            for candidate in item["retrieval_list"]
        ]

        # Re-rank the 20 candidates for the current question
        reranked_nodes = reranker.postprocess_nodes(
            nodes,
            query_bundle=QueryBundle(query_str=item["query"])
        )

        # Save the top-10 reranked passages
        retrieval_list = []

        for node in reranked_nodes:
            retrieval_list.append(
                {
                    "text": node.node.get_content(),
                    "score": float(node.get_score())
                }
            )

        result = dict(item)
        result["retrieval_list"] = retrieval_list

        reranked_results.append(result)

    # Save all reranked results
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(
            reranked_results,
            f,
            ensure_ascii=False,
            indent=2
        )

    print("Saved reranked results to:", output_file)

config.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Reranking queries:   0%|          | 0/2556 [00:00<?, ?it/s]

Saved reranked results to: output/hybrid_bge_reranker_top10.json


### 13.7 – Verify Reranked Output

In [30]:
# T13.7: Verify the reranked result file

import json
import os

reranked_file = "output/hybrid_bge_reranker_top10.json"

assert os.path.exists(reranked_file), "Reranked output file not found."

with open(reranked_file, "r", encoding="utf-8") as f:
    reranked_results = json.load(f)

print("Output file exists:", os.path.exists(reranked_file))
print("Number of query records:", len(reranked_results))
print(
    "Reranked passages for first query:",
    len(reranked_results[0]["retrieval_list"])
)

assert len(reranked_results) == 2556
assert len(reranked_results[0]["retrieval_list"]) == 10

print("BGE reranking output verified successfully.")

Output file exists: True
Number of query records: 2556
Reranked passages for first query: 10
BGE reranking output verified successfully.


### 13.8 – Evaluate Reranked Retrieval

In [31]:
# Evaluate the reranked retrieval results

!python retrieval_evaluate.py \
    --file output/hybrid_bge_reranker_top10.json

Evaluate file: output/hybrid_bge_reranker_top10.json
For file: output/hybrid_bge_reranker_top10.json
Evaluating 2255 valid queries.
Hits@10: 0.9020
Hits@4: 0.7778
MAP@10: 0.3465
MRR@10: 0.6752
--------------------


### 13.9 – Compare Before vs. After Reranking

In [32]:
# Compare hybrid retrieval before and after BGE reranking

pre_rerank_metrics = evaluate(
    "output/hybrid_rrf_top20pool_top10_baseline.json"
)

reranked_metrics = evaluate(
    "output/hybrid_bge_reranker_top10.json"
)

print(
    f"\n{'Method':35s} "
    f"{'Hits@10':>8s} "
    f"{'Hits@4':>8s} "
    f"{'MRR@10':>8s}"
)

for name, metrics in [
    ("Hybrid before reranking", pre_rerank_metrics),
    ("Hybrid + bge-reranker-large", reranked_metrics),
]:
    print(
        f"{name:35s} "
        f"{metrics['Hits@10']:8.4f} "
        f"{metrics['Hits@4']:8.4f} "
        f"{metrics['MRR@10']:8.4f}"
    )

print("\nImprovement after reranking:")
print(
    "Hits@10:",
    f"{reranked_metrics['Hits@10'] - pre_rerank_metrics['Hits@10']:+.4f}"
)
print(
    "Hits@4:",
    f"{reranked_metrics['Hits@4'] - pre_rerank_metrics['Hits@4']:+.4f}"
)
print(
    "MRR@10:",
    f"{reranked_metrics['MRR@10'] - pre_rerank_metrics['MRR@10']:+.4f}"
)


Method                               Hits@10   Hits@4   MRR@10
Hybrid before reranking               0.8647   0.7162   0.5653
Hybrid + bge-reranker-large           0.9020   0.7778   0.6752

Improvement after reranking:
Hits@10: +0.0373
Hits@4: +0.0616
MRR@10: +0.1099


## T13 – Result

Adding `bge-reranker-large` improved retrieval performance across all evaluated metrics.

Hits@10 increased from **0.8647** to **0.9020**, Hits@4 increased from **0.7162** to **0.7778**, and MRR@10 increased from **0.5653** to **0.6752**.

The largest improvement was observed in MRR@10, indicating that the reranker was particularly effective at moving relevant evidence closer to the top of the retrieval ranking.

Overall, the results confirm that adding the BGE reranker improves both retrieval coverage and the ranking quality of relevant evidence.

##T14 – Question Decomposition into Simpler Sub-questions

MultiHop-RAG contains complex questions that may require information from several documents. A single retrieval query may therefore fail to retrieve all the evidence required to answer the original question.

To address this, complex questions are decomposed into simpler sub-questions. Each sub-question focuses on one part of the reasoning process and is used separately for retrieval. The retrieved evidence from the individual sub-questions is then combined so that the system has access to information from multiple relevant documents.

**Experimental design:** the only factor we change is *how the queries are written* (original question vs. original + sub-questions). Everything else stays fixed: the same chunks, retrievers, RRF fusion, pool size (20) and reranker. All comparisons use the **validation** split. The test split is used once at the end.

Assigned to: Rahma Nouri

### 14.1 – Setup: data, train/validation/test split and helper functions

**Why a split?** If we try several settings and pick the one with the best score on *all* questions, that score is too optimistic (like seeing the exam before the exam). So we split the 2,556 questions **once**, with a fixed seed:

- **train (20%)**: not used in Milestone 2. It is kept for Milestone 3 (for example, to tune an "I don't know" threshold).
- **validation (20%)**: used to **compare and choose** settings.
- **test (60%)**: used **once**, only for the final numbers.

The split is *stratified*: every question type (inference, comparison, temporal, null) has the same share in each part. It is saved to `dataset/splits.json` so that every team member uses exactly the same questions.

We also define two helper functions:
- `evaluate_records()` uses the repository's own metric code (`calculate_metrics`) but lets us pick a split and a question type.
- `make_record()` saves results in the same format as the repository, so `retrieval_evaluate.py` still works on our files.



In [33]:
# T14.1: Setup – paths, data, split and helper functions

%cd /content/MultiHop-RAG

!pip install -q bitsandbytes accelerate

import json, os, re, time, random
from collections import defaultdict, Counter

import numpy as np
import torch
from tqdm.auto import tqdm

from retrieval_evaluate import calculate_metrics

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

PERSIST_DIR  = "storage/bge-large-en-v1.5_chunk256"   # dense index built in Milestone 1 (T8)
EMBED_MODEL  = "BAAI/bge-large-en-v1.5"
RERANK_MODEL = "BAAI/bge-reranker-large"
LLM_NAME     = "Qwen/Qwen2.5-3B-Instruct"             # small open LLM, loaded in 4-bit (T14.3)
POOL = 20                                              # candidates given to the reranker (same as T13)
RRF_K = 60                                             # RRF constant (same as T12/T13)

os.makedirs("output", exist_ok=True)

with open("dataset/MultiHopRAG.json", encoding="utf-8") as f:
    questions = json.load(f)
print("Questions:", len(questions))

# ---- train / validation / test split (created once, then re-used) ----
SPLIT_FILE = "dataset/splits.json"
if os.path.exists(SPLIT_FILE):
    with open(SPLIT_FILE) as f:
        splits = json.load(f)
    print("Loaded existing split from", SPLIT_FILE)
else:
    rng = random.Random(SEED)
    by_type = defaultdict(list)
    for i, q in enumerate(questions):
        by_type[q["question_type"]].append(i)
    splits = {"train": [], "val": [], "test": []}
    for qtype in sorted(by_type):
        idxs = by_type[qtype]
        rng.shuffle(idxs)
        n = int(0.2 * len(idxs))
        splits["train"] += idxs[:n]
        splits["val"]   += idxs[n:2 * n]
        splits["test"]  += idxs[2 * n:]
    with open(SPLIT_FILE, "w") as f:
        json.dump(splits, f)
    print("Created new split and saved it to", SPLIT_FILE)

for name, idxs in splits.items():
    counts = Counter(questions[i]["question_type"] for i in idxs)
    print(f"{name:5s} {len(idxs):5d}  {dict(sorted(counts.items()))}")

QTYPES = ["inference_query", "comparison_query", "temporal_query"]   # null_query has no gold evidence

# ---- helper functions ----
def make_record(i, ranked_ids, scores=None, extra=None):
    """One result in the repository's format (query, question_type, gold_list, retrieval_list)."""
    q = questions[i]
    if scores is None:                       # no score available -> use the rank as a score
        scores = [float(len(ranked_ids) - r) for r in range(len(ranked_ids))]
    rec = {
        "query": q["query"],
        "answer": q["answer"],
        "question_type": q["question_type"],
        "gold_list": q["evidence_list"],
        "retrieval_list": [{"text": chunk_texts[c], "score": float(s)}
                           for c, s in zip(ranked_ids, scores)],
    }
    if extra:
        rec.update(extra)
    return rec

def save_records(records, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False)
    print("Saved:", path)

def load_records(path):
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    assert len(data) == len(questions), f"{path} has {len(data)} records"
    for i, d in enumerate(data):
        assert d["query"] == questions[i]["query"], f"order mismatch at {i} in {path}"
    return data

def evaluate_records(records, split=None, qtype=None):
    """Hits@10, Hits@4, MAP@10, MRR@10 with the repository's metric code.
    split = None (all questions), 'train', 'val' or 'test'.  Null queries are skipped, as in M1."""
    ids = set(range(len(questions))) if split is None else set(splits[split])
    rows = [r for i, r in enumerate(records)
            if i in ids and r["question_type"] != "null_query"
            and (qtype is None or r["question_type"] == qtype)]
    retrieved = [[m["text"] for m in r["retrieval_list"]] for r in rows]
    gold = [[m["fact"] for m in r["gold_list"]] for r in rows]
    return calculate_metrics(retrieved, gold)

def print_table(title, results, keys=("Hits@10", "Hits@4", "MRR@10")):
    print(f"\n{title}")
    print(f"{'Experiment':55s}" + "".join(f"{k:>9s}" for k in keys))
    for name, m in results.items():
        print(f"{name:55s}" + "".join(f"{m[k]:9.4f}" for k in keys))

timings = {}   # seconds per stage -> used for the speed/accuracy discussion (and Milestone 3)

/content/MultiHop-RAG
Questions: 2556
Created new split and saved it to dataset/splits.json
train   510  {'comparison_query': 171, 'inference_query': 163, 'null_query': 60, 'temporal_query': 116}
val     510  {'comparison_query': 171, 'inference_query': 163, 'null_query': 60, 'temporal_query': 116}
test   1536  {'comparison_query': 514, 'inference_query': 490, 'null_query': 181, 'temporal_query': 351}


### 14.2 – Load the chunks and build our own dense and BM25 retrievers

To test question decomposition, we have to retrieve evidence for **new** queries (the sub-questions). Calling the repository scripts thousands of times would be very slow: dense retrieval took about 25 minutes for 2,556 queries in T8.

So we build two fast retrievers over **the same 8,034 chunks** from the Milestone 1 index:

- **Dense:** we re-use the chunk embeddings stored in the Milestone 1 index (`bge-large-en-v1.5`). A query is embedded once, and its similarity to all chunks is a single matrix multiplication on the GPU.
- **BM25:** the Okapi BM25 formula (k1 = 1.5, b = 0.75, as in `rank_bm25`), written as one sparse matrix. Scoring many queries is then a single matrix product.

Each chunk text keeps its header (title, source, publication date), as in Milestone 1.

**Sanity check:** before trusting these retrievers, we run them on the original questions and compare with the Milestone 1 numbers. They should be close but not identical, because the tokenization and the query instruction differ slightly.

In [34]:
# T14.2: Load the 8,034 chunks of the Milestone 1 index and build dense + BM25 retrievers

from llama_index import StorageContext
from llama_index.schema import MetadataMode
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
import scipy.sparse as sp

t0 = time.perf_counter()
storage = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
node_dict = storage.docstore.docs
node_ids = list(node_dict.keys())

# Text used by BM25, the reranker and the LLM: header (title / source / date) + chunk text
chunk_texts = [node_dict[n].get_content(metadata_mode=MetadataMode.LLM) for n in node_ids]
print("Number of chunks:", len(chunk_texts))
print("\nExample chunk:\n", chunk_texts[0][:500])

# ---- Dense retriever ----
embedder = SentenceTransformer(EMBED_MODEL, device="cuda")
embedder.half()   # fp16 -> faster on the T4

try:
    chunk_emb = np.array([storage.vector_store.get(n) for n in node_ids], dtype=np.float32)
    print("\nRe-using the chunk embeddings stored in the Milestone 1 index.")
except Exception as e:
    print("\nStored embeddings not available (", repr(e), ") -> computing them again ...")
    embed_texts = [node_dict[n].get_content(metadata_mode=MetadataMode.EMBED) for n in node_ids]
    chunk_emb = embedder.encode(embed_texts, batch_size=64, normalize_embeddings=True,
                                show_progress_bar=True, convert_to_numpy=True)

chunk_emb = chunk_emb / np.linalg.norm(chunk_emb, axis=1, keepdims=True)   # cosine similarity
chunk_emb_t = torch.tensor(chunk_emb, device="cuda", dtype=torch.float16)

# BGE models expect this instruction in front of a search query (LlamaIndex adds the same one)
QUERY_INSTRUCTION = "Represent this question for searching relevant passages: "

def dense_search(queries, k=POOL, batch=1024):
    """Return, for every query, the indices of the k most similar chunks."""
    results = []
    for s in range(0, len(queries), batch):
        q = embedder.encode([QUERY_INSTRUCTION + x for x in queries[s:s + batch]],
                            batch_size=64, normalize_embeddings=True, convert_to_tensor=True)
        scores = q.half() @ chunk_emb_t.T
        top = torch.topk(scores, k, dim=1).indices.cpu().numpy()
        results += [row.tolist() for row in top]
    return results

# ---- BM25 retriever (Okapi BM25 as one sparse matrix) ----
vectorizer = CountVectorizer(lowercase=True, token_pattern=r"(?u)\b\w+\b")
tf = vectorizer.fit_transform(chunk_texts).tocsr().astype(np.float32)   # chunks x words (counts)
N = tf.shape[0]
doc_len = np.asarray(tf.sum(axis=1)).ravel()
avgdl = doc_len.mean()
df = np.bincount(tf.indices, minlength=tf.shape[1])          # in how many chunks each word appears
idf = np.log(1 + (N - df + 0.5) / (df + 0.5))
K1, B = 1.5, 0.75
row_of_value = np.repeat(np.arange(N), np.diff(tf.indptr))
weights = idf[tf.indices] * tf.data * (K1 + 1) / (tf.data + K1 * (1 - B + B * doc_len[row_of_value] / avgdl))
bm25_matrix = sp.csr_matrix((weights, tf.indices, tf.indptr), shape=tf.shape)

def bm25_search(queries, k=POOL, batch=512):
    """BM25 score(query, chunk) = sum of the chunk's BM25 weights for the query words."""
    results = []
    for s in range(0, len(queries), batch):
        q = vectorizer.transform(queries[s:s + batch])          # word counts of each query
        scores = (q @ bm25_matrix.T).toarray()                 # queries x chunks
        top = np.argsort(-scores, axis=1)[:, :k]
        results += [row.tolist() for row in top]
    return results

timings["T14 build retrievers"] = time.perf_counter() - t0

# ---- Retrieve for the ORIGINAL questions (needed for the baseline and the decomposition run) ----
orig_queries = [q["query"] for q in questions]

t0 = time.perf_counter()
orig_dense = dense_search(orig_queries)
timings["T14 dense retrieval, original (2556 q)"] = time.perf_counter() - t0

t0 = time.perf_counter()
orig_bm25 = bm25_search(orig_queries)
timings["T14 BM25 retrieval, original (2556 q)"] = time.perf_counter() - t0

rec_dense = [make_record(i, orig_dense[i][:10]) for i in range(len(questions))]
rec_bm25  = [make_record(i, orig_bm25[i][:10])  for i in range(len(questions))]

# ---- Sanity check against Milestone 1 (all 2,255 answerable questions) ----
M1 = {"Dense (Milestone 1 script)": {"Hits@10": 0.7996, "Hits@4": 0.6058, "MRR@10": 0.4372},
      "BM25 (Milestone 1 script)":  {"Hits@10": 0.8683, "Hits@4": 0.7126, "MRR@10": 0.5747}}
check = {**M1,
         "Dense (our re-implementation)": evaluate_records(rec_dense),
         "BM25 (our re-implementation)":  evaluate_records(rec_bm25)}
print_table("Sanity check – all answerable questions", check)
print("\nSmall differences are expected (tokenization / query instruction). Large differences mean a bug.")

Number of chunks: 8034

Example chunk:
 [Excerpt from document]
title: 200+ of the best deals from Amazon's Cyber Monday sale
published_at: 2023-11-27T08:45:59+00:00
source: Mashable
Excerpt:
-----
Table of Contents Table of Contents Echo, Fire TV, and Kindle deals Apple deals TV deals Laptop deals Headphone and earbud deals Tablet deals Gaming deals Speaker deals Vacuum deals Kitchen deals Smart home deals Fitness deals Beauty tech deals Drone deals Camera deals Lego deals Gift card deals

UPDATE: Nov. 27, 2023, 5:00 a.m. EST This po


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]


Re-using the chunk embeddings stored in the Milestone 1 index.

Sanity check – all answerable questions
Experiment                                               Hits@10   Hits@4   MRR@10
Dense (Milestone 1 script)                                0.7996   0.6058   0.4372
BM25 (Milestone 1 script)                                 0.8683   0.7126   0.5747
Dense (our re-implementation)                             0.7991   0.6027   0.4371
BM25 (our re-implementation)                              0.8962   0.7424   0.6059

Small differences are expected (tokenization / query instruction). Large differences mean a bug.


### 14.3 – Load a small open LLM in 4-bit precision

We use **Qwen2.5-3B-Instruct** for two jobs: splitting questions here in T14, and answering them in T15.

**What is 4-bit?** Normally each model weight is stored with 16 bits. With 4-bit quantization (`bitsandbytes`, NF4 format), each weight uses only 4 bits, so the model needs about 4 times less GPU memory. A 3-billion-parameter model then fits in about 2–3 GB, which is easy on a free Colab T4. The price is a small loss in quality and somewhat slower generation.

Two helper functions are defined here and re-used in T15:
- `generate_batch()` generates text for several prompts at once. Batching is much faster than one prompt at a time.
- `run_cached()` saves the answers to a file after **every** batch. If Colab disconnects, re-running the cell continues where it stopped instead of starting again.

We use **greedy decoding** (`do_sample=False`), so the outputs are deterministic and reproducible.

In [35]:
# T14.3: Load Qwen2.5-3B-Instruct in 4-bit and define generation helpers (also used in T15)

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

t0 = time.perf_counter()
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",              # 4-bit "NormalFloat" format
    bnb_4bit_compute_dtype=torch.float16,   # calculations are done in fp16
    bnb_4bit_use_double_quant=True,         # saves a little more memory
)
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
tokenizer.padding_side = "left"             # required for batched generation with decoder-only models
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, quantization_config=bnb_config, device_map="auto")
llm.eval()
timings["T14 load 4-bit LLM"] = time.perf_counter() - t0

print(f"LLM memory footprint: {llm.get_memory_footprint() / 1e9:.2f} GB")
print(f"GPU memory allocated in total: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

def chat_prompt(messages):
    """Turn a list of chat messages into the text format the model was trained on."""
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def generate_batch(prompts, max_new_tokens):
    enc = tokenizer(prompts, return_tensors="pt", padding=True).to(llm.device)
    out = llm.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=PAD_ID)
    new_tokens = out[:, enc["input_ids"].shape[1]:]          # keep only the generated part
    return tokenizer.batch_decode(new_tokens, skip_special_tokens=True)

def run_cached(prompts, cache_path, max_new_tokens, batch_size=16, desc=""):
    """prompts: {question_index: prompt}.  Returns {question_index: generated text}.
    Saves after every batch, so an interrupted run can be resumed."""
    cache = {}
    if os.path.exists(cache_path):
        with open(cache_path) as f:
            cache = {int(k): v for k, v in json.load(f).items()}
    todo = [i for i in prompts if i not in cache]
    todo.sort(key=lambda i: len(prompts[i]))                 # similar lengths together -> less padding
    print(f"{desc}: {len(prompts) - len(todo)} cached, {len(todo)} to generate")
    for s in tqdm(range(0, len(todo), batch_size), desc=desc):
        batch = todo[s:s + batch_size]
        for i, text in zip(batch, generate_batch([prompts[i] for i in batch], max_new_tokens)):
            cache[i] = text.strip()
        with open(cache_path, "w") as f:
            json.dump(cache, f)
    return {i: cache[i] for i in prompts}

# quick test
print(generate_batch([chat_prompt([{"role": "user", "content": "Say hello in one word."}])], 10))

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLM memory footprint: 2.01 GB
GPU memory allocated in total: 5.00 GB
['Hello.']


### 14.4 – Split every question into simpler sub-questions

**Idea:** a multi-hop question needs evidence from 2–4 different articles. A single query often finds one article but misses the other. Example:

> *"Did the TechCrunch report on X come out before The Verge report on Y?"*

splits into

> *"What did TechCrunch report about X?"*
> *"What did The Verge report about Y?"*

Each sub-question is "about one article", so each one should find its own evidence.

The prompt asks the LLM for 1–3 sub-questions, one per line. It must copy names of sources, people, companies and dates exactly, because BM25 relies on these exact words. We give it four **invented** examples (few-shot prompting). They are not taken from the dataset, so no test information leaks into the prompt. If the output cannot be parsed, we fall back to the original question.

We decompose all 2,556 questions once and cache the result in `output/t14_subquestions_raw.json`. On a T4 this takes roughly 20–40 minutes.

In [36]:
# T14.4: Decompose each question into 1–3 sub-questions with the 4-bit LLM

DECOMP_SYSTEM = """You split complex questions about news articles into simple search queries.
Rules:
- Write 1 to 3 short sub-questions, one per line, without numbering.
- Each sub-question must be answerable from ONE news article.
- Copy the names of news sources (for example TechCrunch, The Verge), people, companies and dates exactly.
- If the question is already simple, repeat it unchanged.
- Output only the sub-questions."""

# Invented examples (not from the dataset) that show the expected format
FEW_SHOT = [
    ("Did the Reuters report about Nova Motors' battery recall come out before the Wired report about Nova Motors' new factory?",
     "What did Reuters report about Nova Motors' battery recall?\nWhat did Wired report about Nova Motors' new factory?"),
    ("Which streaming company, according to an article from Variety and an article from The Hollywood Reporter, raised its prices in October and lost two major sports contracts?",
     "Which streaming company raised its prices in October according to Variety?\nWhich streaming company lost two major sports contracts according to The Hollywood Reporter?"),
    ("Do both the Engadget article and the Ars Technica article say that the new Lumo phone has a better camera than the previous model?",
     "What does Engadget say about the camera of the new Lumo phone?\nWhat does Ars Technica say about the camera of the new Lumo phone?"),
    ("What did Fortune report about the CEO of Brightline Energy stepping down?",
     "What did Fortune report about the CEO of Brightline Energy stepping down?"),
]

def build_decomp_prompt(question):
    messages = [{"role": "system", "content": DECOMP_SYSTEM}]
    for q, a in FEW_SHOT:
        messages += [{"role": "user", "content": q}, {"role": "assistant", "content": a}]
    messages.append({"role": "user", "content": question})
    return chat_prompt(messages)

def parse_subquestions(text, original, max_n=3):
    """One sub-question per line; remove numbering/bullets; fall back to the original question."""
    subs = []
    for line in text.split("\n"):
        line = re.sub(r"^\s*(\d+[\.\)]|[-*•])\s*", "", line).strip().strip('"').strip()
        if len(line) > 10 and line not in subs:
            subs.append(line)
    return subs[:max_n] if subs else [original]

t0 = time.perf_counter()
raw = run_cached({i: build_decomp_prompt(q["query"]) for i, q in enumerate(questions)},
                 "output/t14_subquestions_raw.json", max_new_tokens=160, batch_size=16,
                 desc="Decomposing questions")
timings["T14 decomposition (LLM, 2556 q)"] = time.perf_counter() - t0

sub_questions = [parse_subquestions(raw[i], questions[i]["query"]) for i in range(len(questions))]

with open("output/t14_subquestions.json", "w", encoding="utf-8") as f:
    json.dump([{"query": questions[i]["query"], "question_type": questions[i]["question_type"],
                "sub_questions": sub_questions[i]} for i in range(len(questions))],
              f, ensure_ascii=False, indent=2)

n_subs = [len(s) for s in sub_questions]
unchanged = sum(s == [questions[i]["query"]] for i, s in enumerate(sub_questions))
print("Average number of sub-questions:", round(float(np.mean(n_subs)), 2))
print("Distribution:", dict(sorted(Counter(n_subs).items())))
print(f"Questions left unchanged: {unchanged} ({unchanged / len(questions):.1%})")

Decomposing questions: 0 cached, 2556 to generate


Decomposing questions:   0%|          | 0/160 [00:00<?, ?it/s]

Average number of sub-questions: 2.32
Distribution: {1: 30, 2: 1669, 3: 857}
Questions left unchanged: 4 (0.2%)


### 14.5 – Look at some decompositions

Numbers alone can hide problems, so we read one validation example per question type. Good sub-questions keep the source names and split the question into "one article each" parts. Typical failures are sub-questions that drop the source name or simply repeat the whole question.

In [37]:
# T14.5: Inspect one validation example per question type

rng = random.Random(SEED)
for qtype in QTYPES + ["null_query"]:
    i = rng.choice([j for j in splits["val"] if questions[j]["question_type"] == qtype])
    print(f"[{qtype}] {questions[i]['query']}")
    for s in sub_questions[i]:
        print("   ->", s)
    print()

[inference_query] What company spends billions to maintain its default search engine status on various platforms, influences local search rankings with specific criteria, and is accused by news publishers of anticompetitive practices that harm their revenue, according to articles from The Verge and TechCrunch?
   -> What company spends billions to maintain its default search engine status on various platforms?
   -> According to The Verge, what specific criteria does this company use to influence local search rankings?
   -> According to TechCrunch, what anticompetitive practices has this company been accused of by news publishers?

[comparison_query] Does the 'Fortune' article suggest that Sam Bankman-Fried used Caroline Ellison as a front for illicit activities involving customer funds at Alameda Research, while the 'TechCrunch' articles focus on his alleged intentions to use wealth for altruistic purposes and accusations of committing fraud for personal gain, without mentioning the 

### 14.6 – Retrieve with the original question vs. with sub-questions (hybrid RRF)

We build two candidate pools of the **same size** (20 chunks) for every question:

| Pool | Ranked lists fused with RRF |
|---|---|
| **Original** (T14 baseline) | dense(question), BM25(question) |
| **Decomposed** | dense(question), BM25(question) **+** dense(sub-q), BM25(sub-q) for every sub-question |

**Reciprocal Rank Fusion (RRF):** a chunk at rank *r* in a list gets 1/(60 + r) points, and the points from all lists are added together. A chunk that appears high in several lists wins. This is the same method and constant (k = 60) as in T12/T13.

**Why the same pool size?** If the decomposed pool were bigger, it could look better just because it has more chunks. With equal pool sizes, the **only** difference between the two runs is how the queries were written.

We keep the original question in the decomposed pool as a safety net, in case the LLM writes a bad sub-question.

In [38]:
# T14.6: Hybrid RRF pools for original vs. decomposed queries

def rrf_fuse(ranked_lists, k_rrf=RRF_K, top=POOL):
    scores = defaultdict(float)
    for lst in ranked_lists:
        for rank, c in enumerate(lst):
            scores[c] += 1.0 / (k_rrf + rank + 1)
    best = sorted(scores, key=scores.get, reverse=True)[:top]
    return best, [scores[c] for c in best]

# Retrieve for all sub-questions in one go (flattened list + mapping back to the question)
flat_subs, owner = [], []
for i, subs in enumerate(sub_questions):
    for s in subs:
        flat_subs.append(s)
        owner.append(i)
print("Total sub-questions:", len(flat_subs))

t0 = time.perf_counter()
sub_dense = dense_search(flat_subs)
sub_bm25 = bm25_search(flat_subs)
timings[f"T14 dense+BM25 retrieval, sub-questions ({len(flat_subs)} q)"] = time.perf_counter() - t0

lists_per_question = defaultdict(list)
for j, i in enumerate(owner):
    lists_per_question[i] += [sub_dense[j], sub_bm25[j]]

orig_pool, decomp_pool = [], []
for i in range(len(questions)):
    orig_pool.append(rrf_fuse([orig_dense[i], orig_bm25[i]]))
    decomp_pool.append(rrf_fuse([orig_dense[i], orig_bm25[i]] + lists_per_question[i]))

# Records before reranking: top-10 of each pool
rec_orig_hybrid = [make_record(i, orig_pool[i][0][:10], orig_pool[i][1][:10]) for i in range(len(questions))]
rec_decomp_hybrid = [make_record(i, decomp_pool[i][0][:10], decomp_pool[i][1][:10],
                                 extra={"sub_questions": sub_questions[i]}) for i in range(len(questions))]
save_records(rec_orig_hybrid, "output/t14_orig_hybrid_top10.json")
save_records(rec_decomp_hybrid, "output/t14_decomp_hybrid_top10.json")

# How different are the two pools?
overlap = np.mean([len(set(orig_pool[i][0]) & set(decomp_pool[i][0])) / POOL for i in range(len(questions))])
print(f"Average overlap between the two 20-chunk pools: {overlap:.1%}")

Total sub-questions: 5939
Saved: output/t14_orig_hybrid_top10.json
Saved: output/t14_decomp_hybrid_top10.json
Average overlap between the two 20-chunk pools: 75.2%


### 14.7 – Re-rank both pools with `bge-reranker-large`

As in T13, the cross-encoder reranker reads **(original question, chunk)** pairs and gives each pair a relevance score. We keep the 10 best chunks.

**Speed-up compared with T13:**
1. `use_fp16=True`: half precision runs much faster on a T4 GPU.
2. All 51,120 pairs (2,556 questions × 20 chunks) are scored in **one** call, so the reranker can form full batches.

In T13 the reranking took about 2 hours. Here it should take minutes.

In [39]:
# T14.7: Rerank the original and the decomposed pools with bge-reranker-large (fp16)

from FlagEmbedding import FlagReranker

reranker = FlagReranker(RERANK_MODEL, use_fp16=True)

def rerank(pools, desc, top_n=10):
    pairs, sizes = [], []
    for i, (cands, _) in enumerate(pools):
        pairs += [[questions[i]["query"], chunk_texts[c]] for c in cands]
        sizes.append(len(cands))
    t0 = time.perf_counter()
    scores = reranker.compute_score(pairs, batch_size=64)
    timings[f"T14 rerank {desc} ({len(pairs)} pairs)"] = time.perf_counter() - t0
    out, pos = [], 0
    for (cands, _), n in zip(pools, sizes):
        s = np.array(scores[pos:pos + n]); pos += n
        order = np.argsort(-s)[:top_n]
        out.append(([cands[j] for j in order], [float(s[j]) for j in order]))
    return out

orig_reranked = rerank(orig_pool, "original pool")
decomp_reranked = rerank(decomp_pool, "decomposed pool")

rec_orig_rerank = [make_record(i, *orig_reranked[i]) for i in range(len(questions))]
rec_decomp_rerank = [make_record(i, *decomp_reranked[i], extra={"sub_questions": sub_questions[i]})
                     for i in range(len(questions))]
save_records(rec_orig_rerank, "output/t14_orig_pool20_rerank.json")
save_records(rec_decomp_rerank, "output/t14_decomp_pool20_rerank.json")

# Sanity check: our original+rerank pipeline should be close to T13 (Hits@10 0.9020 on all questions)
print_table("Sanity check vs. T13 – all answerable questions", {
    "T13: hybrid + bge-reranker-large (repo scripts)": {"Hits@10": 0.9020, "Hits@4": 0.7778, "MRR@10": 0.6751},
    "T14: original pool + reranker (our pipeline)": evaluate_records(rec_orig_rerank),
})

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Compute Scores: 100%|██████████| 799/799 [05:20<00:00,  2.49it/s]


Saved: output/t14_orig_pool20_rerank.json
Saved: output/t14_decomp_pool20_rerank.json

Sanity check vs. T13 – all answerable questions
Experiment                                               Hits@10   Hits@4   MRR@10
T13: hybrid + bge-reranker-large (repo scripts)           0.9020   0.7778   0.6751
T14: original pool + reranker (our pipeline)              0.9091   0.7849   0.6819


### 14.8 – Compare on the validation set and choose the retrieval setting

All six variants are evaluated on the **validation** questions only. One factor changes between neighbouring rows:

- rows 1–3: dense vs. BM25 vs. hybrid (original question)
- row 3 → 5 and row 4 → 6: **original vs. decomposed queries** (the T14 change), without and with the reranker
- row 3 → 4 and row 5 → 6: effect of the reranker

We also break the results down by question type, because decomposition should help most on questions that clearly involve two articles (comparison, temporal).

**Selection rule (decided before looking at the numbers):** between the two reranked settings, we choose the one with the higher **Hits@4** on validation, with MRR@10 as a tie-breaker. The reason is that T15 gives the LLM the top-4 chunks, so Hits@4 is the number that matters most for answering.

In [44]:
# T14.8: Validation results, per-type breakdown and selection

T14_EXPERIMENTS = {
    "R1 Dense, original question":                 rec_dense,
    "R2 BM25, original question":                  rec_bm25,
    "R3 Hybrid RRF top-10, original":              rec_orig_hybrid,
    "R4 Hybrid pool-20 + reranker, original":      rec_orig_rerank,
    "R5 Hybrid RRF top-10, decomposed":            rec_decomp_hybrid,
    "R6 Hybrid pool-20 + reranker, decomposed":    rec_decomp_rerank,
}
n_val = sum(questions[i]["question_type"] != "null_query" for i in splits["val"])
val_results = {name: evaluate_records(rec, "val") for name, rec in T14_EXPERIMENTS.items()}
print_table(f"T14 – VALIDATION set ({n_val} answerable questions)", val_results,
            keys=("Hits@10", "Hits@4", "MRR@10", "MAP@10"))

# Per question type: does decomposition help some types more than others?
print(f"\n{'Question type (val)':20s}{'n':>5s} | {'Hits@10 orig':>12s}{'Hits@10 dec':>12s} | {'MRR orig':>9s}{'MRR dec':>9s}")
for t in QTYPES:
    n = sum(questions[i]["question_type"] == t for i in splits["val"])
    a = evaluate_records(rec_orig_rerank, "val", t)
    b = evaluate_records(rec_decomp_rerank, "val", t)
    print(f"{t:20s}{n:5d} | {a['Hits@10']:12.4f}{b['Hits@10']:12.4f} | {a['MRR@10']:9.4f}{b['MRR@10']:9.4f}")

# Selection on validation only
candidates = {
    "R4 Hybrid pool-20 + reranker, original":   "output/t14_orig_pool20_rerank.json",
    "R6 Hybrid pool-20 + reranker, decomposed": "output/t14_decomp_pool20_rerank.json",
}
BEST_T14 = max(candidates, key=lambda n: (val_results[n]["Hits@4"], val_results[n]["MRR@10"]))
BEST_T14_FILE = candidates[BEST_T14]
print(f"\nSelected on validation: {BEST_T14}  ->  {BEST_T14_FILE}")


T14 – VALIDATION set (450 answerable questions)
Experiment                                               Hits@10   Hits@4   MRR@10   MAP@10
R1 Dense, original question                               0.8156   0.6244   0.4565   0.2157
R2 BM25, original question                                0.8956   0.7378   0.6210   0.3206
R3 Hybrid RRF top-10, original                            0.8844   0.7311   0.5861   0.2941
R4 Hybrid pool-20 + reranker, original                    0.9156   0.7978   0.7029   0.3646
R5 Hybrid RRF top-10, decomposed                          0.9200   0.7000   0.5083   0.2572
R6 Hybrid pool-20 + reranker, decomposed                  0.9311   0.8111   0.7157   0.3704

Question type (val)     n | Hits@10 orig Hits@10 dec |  MRR orig  MRR dec
inference_query       163 |       0.9509      0.9448 |    0.7889   0.8001
comparison_query      171 |       0.9064      0.9357 |    0.7155   0.7278
temporal_query        116 |       0.8793      0.9052 |    0.5635   0.5794

Selected 

### 14.9 – Final evaluation on the test set (run once, after the choice above)

The selection in 14.8 has now been made, so we can look at the test set. We report the Milestone 1 dense baseline, the T14 baseline (original question + reranker), and the decomposed version. All results and timings are saved to `output/t14_results.json`.

In [45]:
# T14.9: Test-set results (only after the validation-based choice in 14.8)

n_test = sum(questions[i]["question_type"] != "null_query" for i in splits["test"])
test_results = {name: evaluate_records(T14_EXPERIMENTS[name], "test") for name in [
    "R1 Dense, original question",
    "R4 Hybrid pool-20 + reranker, original",
    "R6 Hybrid pool-20 + reranker, decomposed",
]}
print_table(f"T14 – TEST set ({n_test} answerable questions)", test_results,
            keys=("Hits@10", "Hits@4", "MRR@10", "MAP@10"))
print("\nSelected setting (chosen on validation):", BEST_T14)

print("\nTime per stage:")
for k, v in timings.items():
    print(f"  {k:60s} {v:8.1f} s")

with open("output/t14_results.json", "w") as f:
    json.dump({"validation": val_results, "test": test_results, "selected": BEST_T14,
               "selected_file": BEST_T14_FILE, "timings_seconds": timings,
               "settings": {"pool": POOL, "rrf_k": RRF_K, "llm": LLM_NAME,
                            "reranker": RERANK_MODEL, "split_seed": SEED}}, f, indent=2)
print("\nSaved: output/t14_results.json")


T14 – TEST set (1355 answerable questions)
Experiment                                               Hits@10   Hits@4   MRR@10   MAP@10
R1 Dense, original question                               0.8074   0.6111   0.4377   0.2054
R4 Hybrid pool-20 + reranker, original                    0.9122   0.7860   0.6788   0.3449
R6 Hybrid pool-20 + reranker, decomposed                  0.9041   0.7734   0.6696   0.3423

Selected setting (chosen on validation): R6 Hybrid pool-20 + reranker, decomposed

Time per stage:
  T14 build retrievers                                             69.5 s
  T14 dense retrieval, original (2556 q)                            4.2 s
  T14 BM25 retrieval, original (2556 q)                             1.4 s
  T14 load 4-bit LLM                                               29.9 s
  T14 decomposition (LLM, 2556 q)                                1156.5 s
  T14 dense+BM25 retrieval, sub-questions (5939 q)                  6.6 s
  T14 rerank original pool (51120 pairs)    

## T14 – Result

**Validation set** (450 answerable questions)

| ID | Retrieval setting | Hits@10 | Hits@4 | MRR@10 |
|---|---|---|---|---|
| R3 | Hybrid RRF, original question | 0.8844 | 0.7311 | 0.5861 |
| R4 | Hybrid pool-20 + reranker, original (T14 baseline) | 0.9156 | 0.7978 | 0.7029 |
| R5 | Hybrid RRF, decomposed | 0.9200 | 0.7000 | 0.5083 |
| R6 | Hybrid pool-20 + reranker, decomposed | **0.9311** | **0.8111** | **0.7157** |

**Selected on validation:** R6.

**Test set** (1355 answerable questions)

| ID | Retrieval setting | Hits@10 | Hits@4 | MRR@10 |
|---|---|---|---|---|
| R1 | Dense, original question (Milestone 1) | 0.8074 | 0.6111 | 0.4377 |
| R4 | Hybrid pool-20 + reranker, original | **0.9122** | **0.7860** | **0.6788** |
| R6 | Hybrid pool-20 + reranker, decomposed (selected) | 0.9041 | 0.7734 | 0.6696 |

**Discussion**
- **Without the reranker (R3 → R5):** decomposition finds more of the gold evidence (Hits@10 +3.6 points) but ranks it worse (Hits@4 −3.1, MRR −7.8 points). The results of 2–3 sub-questions are mixed together, so the best chunk is often not at the top.
- **With the reranker (R4 → R6):** the reranker fixes the order, and decomposition gives a small gain on validation (+1.6 points Hits@10, +1.3 MRR).
- **Per question type (validation):** comparison (+2.9 Hits@10) and temporal (+2.6) questions gain, because they name two different articles and each sub-question finds one of them. Inference questions lose a little (−0.6), because they are often answerable from one article, so splitting them adds noise.
- **On the test set the gain does not hold:** R6 is 0.8 points below R4 in Hits@10. The small validation gain was not reliable. This is why we chose on validation and used the test set only once, at the end.
- **Decomposition quality (14.5):** the LLM kept the names of news sources, people, companies and dates. Almost all questions (98.8%) were split into 2–3 sub-questions (5,939 sub-questions in total). It also split null questions into confident-looking sub-questions, so decomposition alone cannot detect unanswerable questions.
- **Cost (L4 GPU):** decomposition needs one LLM call per question (1157 s for 2,556 questions, ≈0.45 s/question) plus a second reranking run (≈400 s). For this dataset, this extra cost does **not** give a reliable gain over hybrid retrieval + reranker.

## T15 – Generate Answers with a Small Open LLM in 4-bit Precision

After relevant evidence has been retrieved, a small open language model (**Qwen2.5-3B-Instruct**, 4-bit NF4) generates answers to the MultiHop-RAG questions. The question and the retrieved evidence are given to the model, and the model is told to answer **only** from that evidence, or to reply *"Insufficient information."* when the evidence is not enough.

This turns the project from retrieval-only evaluation into a complete RAG pipeline. The generated answers are saved for Milestone 3.

**Experimental design:** we change one factor at a time: no evidence → M1 evidence → best T14 evidence → more chunks. We choose on **validation** and report on **test**.

Assigned to: Rahma Nouri

### 15.1 – Setup: which retrieved evidence goes to the LLM

T15 re-uses the data, split, helper functions and the 4-bit LLM from T14.1 and T14.3. After a Colab restart, run those two cells first. The cached files mean nothing has to be generated twice.

We compare two evidence sources, and **only the evidence changes** between them:

- **M1 dense evidence:** the Milestone 1 dense top-10 (`bge-large-en-v1.5`). This is the *baseline*.
- **Best T14 evidence:** the retrieval setting chosen on validation in T14.8. If T14 was not run, we use the T13 reranked file.

In [46]:
# T15.1: Setup – evidence sources

assert "llm" in globals() and "questions" in globals(), "Run T14.1 and T14.3 first."

BASELINE_EVIDENCE_FILE = "output/bge-large-en-v1.5_retrieval_test.json"      # Milestone 1 dense top-10
BEST_EVIDENCE_FILE = globals().get("BEST_T14_FILE", "output/hybrid_bge_reranker_top10.json")

evidence = {
    "m1_dense": load_records(BASELINE_EVIDENCE_FILE),
    "best":     load_records(BEST_EVIDENCE_FILE),
}
print("Baseline evidence:", BASELINE_EVIDENCE_FILE)
print("Best evidence    :", BEST_EVIDENCE_FILE)

# What do the gold answers look like? (validation set)
val_answers = Counter(questions[i]["answer"] for i in splits["val"])
print("\nMost common gold answers (val):", val_answers.most_common(8))

Baseline evidence: output/bge-large-en-v1.5_retrieval_test.json
Best evidence    : output/t14_decomp_pool20_rerank.json

Most common gold answers (val): [('Yes', 147), ('no', 107), ('Insufficient information.', 60), ('Sam Bankman-Fried', 51), ('Google', 40), ('Sam Altman', 11), ('No', 7), ('Consistent', 7)]


### 15.2 – Prompts

The dataset's answers are short: mostly **"Yes"/"No"**, a name, or **"Insufficient information."** for the null questions. So the prompt asks for a very short answer and gives the model an explicit way to say "I don't know".

- **RAG prompt:** the top-*k* retrieved chunks (each with title, source and date) and the question. The model must use **only** this evidence.
- **Closed-book prompt:** the question alone, with no evidence. It shows what the model "knows" by itself. Most articles are from late 2023, close to or after the model's training data, so we expect weak accuracy and many abstentions or invented answers. This makes the value of retrieval visible.

In [47]:
# T15.2: Prompt templates

ABSTAIN = "Insufficient information."

RAG_SYSTEM = f"""You answer questions about news articles.
Use ONLY the evidence given by the user. Do not use outside knowledge.
Answer in as few words as possible:
- For yes/no questions, answer only "Yes" or "No".
- For other questions, answer only with the name, word or short phrase.
- If the evidence does not contain enough information to answer, answer exactly: {ABSTAIN}"""

CLOSED_SYSTEM = f"""You answer questions about news articles from your own knowledge.
Answer in as few words as possible:
- For yes/no questions, answer only "Yes" or "No".
- For other questions, answer only with the name, word or short phrase.
- If you do not know the answer, answer exactly: {ABSTAIN}"""

def build_rag_prompt(i, source, k):
    docs = evidence[source][i]["retrieval_list"][:k]
    context = "\n\n".join(f"[Document {j + 1}]\n{d['text']}" for j, d in enumerate(docs))
    user = f"Evidence:\n{context}\n\nQuestion: {questions[i]['query']}\nAnswer:"
    return chat_prompt([{"role": "system", "content": RAG_SYSTEM}, {"role": "user", "content": user}])

def build_closed_prompt(i):
    user = f"Question: {questions[i]['query']}\nAnswer:"
    return chat_prompt([{"role": "system", "content": CLOSED_SYSTEM}, {"role": "user", "content": user}])

# Show one full RAG prompt so we know exactly what the model receives
example_i = splits["val"][0]
print(build_rag_prompt(example_i, "best", 4)[:2500])

<|im_start|>system
You answer questions about news articles.
Use ONLY the evidence given by the user. Do not use outside knowledge.
Answer in as few words as possible:
- For yes/no questions, answer only "Yes" or "No".
- For other questions, answer only with the name, word or short phrase.
- If the evidence does not contain enough information to answer, answer exactly: Insufficient information.<|im_end|>
<|im_start|>user
Evidence:
[Document 1]
[Excerpt from document]
title: Inter Miami season tickets: Lionel Messi demand causes 2024 prices soar to most expensive in world
published_at: 2023-10-04T22:12:56+00:00
source: Sporting News
Excerpt:
-----
One of Europe's most expensive tickets can be found at Tottenham, thanks in part to the brand new state-of-the-art Tottenham Hotspur Stadium, and even then its cheapest ticket is around $974, while its most expensive package comes out to $2,498 for the year.

The most expensive premium packages at Inter Miami in 2024, which will run supporters

### 15.3 – How we score the answers

**Accuracy:** an answer is correct if the gold answer appears in the model's first line as a whole word or phrase, after lower-casing and removing punctuation. For example, gold "Yes" matches the prediction "Yes." but not "Yesterday".

For the hallucination question we also measure:
- **Null: abstained:** on the null questions (no answer exists in the corpus), how often the model correctly says *Insufficient information*. Higher is better.
- **Answerable: wrongly abstained:** on answerable questions, how often the model gives up although an answer exists. Lower is better.

These two numbers pull against each other. A model that always says "I don't know" gets 100% on the first and fails the second. Milestone 3 studies this trade-off in detail.

*Limitation:* string matching can mark a correct answer as wrong, for example "Bankman-Fried" vs. gold "Sam Bankman-Fried". Milestone 3 compares this metric with manual checks and an LLM judge.

In [48]:
# T15.3: Answer scoring

def normalize(text):
    text = re.sub(r"[^\w\s]", " ", text.lower())
    return " ".join(text.split())

def is_abstention(pred):
    return "insufficient information" in normalize(pred)

def is_correct(pred, gold):
    g = normalize(gold)
    if g == "insufficient information":          # null question: correct = the model abstains
        return is_abstention(pred)
    if is_abstention(pred):
        return False
    lines = pred.strip().split("\n")
    p = normalize(lines[0]) if lines else ""
    return p == g or re.search(r"\b" + re.escape(g) + r"\b", p) is not None

def answer_metrics(preds, split):
    ids = splits[split]
    answerable = [i for i in ids if questions[i]["question_type"] != "null_query"]
    nulls = [i for i in ids if questions[i]["question_type"] == "null_query"]
    m = {
        "Acc (all)":        np.mean([is_correct(preds[i], questions[i]["answer"]) for i in ids]),
        "Acc (answerable)": np.mean([is_correct(preds[i], questions[i]["answer"]) for i in answerable]),
    }
    for t in QTYPES:
        t_ids = [i for i in ids if questions[i]["question_type"] == t]
        m[f"Acc {t.split('_')[0]}"] = np.mean([is_correct(preds[i], questions[i]["answer"]) for i in t_ids])
    m["Null: abstained"] = np.mean([is_abstention(preds[i]) for i in nulls])
    m["Answerable: wrongly abstained"] = np.mean([is_abstention(preds[i]) for i in answerable])
    return {k: float(v) for k, v in m.items()}

# Small self-test of the scoring function
assert is_correct("Yes.", "Yes") and not is_correct("Yesterday", "Yes")
assert is_correct("Insufficient information.", "Insufficient information.")
assert not is_correct("Insufficient information.", "No")
assert is_correct("It was Sam Bankman-Fried", "Sam Bankman-Fried")
print("Scoring function OK")

ANS_KEYS = ["Acc (all)", "Acc (answerable)", "Acc inference", "Acc comparison", "Acc temporal",
            "Null: abstained", "Answerable: wrongly abstained"]

def print_answer_table(title, results):
    short = ["All", "Answ.", "Infer.", "Compar.", "Tempor.", "Null✓", "FalseAbst"]
    print(f"\n{title}")
    print(f"{'Experiment':42s}" + "".join(f"{s:>10s}" for s in short))
    for name, m in results.items():
        print(f"{name:42s}" + "".join(f"{m[k]:10.4f}" for k in ANS_KEYS))

Scoring function OK


### 15.4 – Generate answers on the validation set

Four experiments. Each one changes a single factor compared with the row before it:

| ID | Evidence | Chunks (*k*) | What it tests |
|---|---|---|---|
| G0 | none (closed-book) | 0 | Does retrieval help at all? |
| G1 | Milestone 1 dense | 4 | **Baseline** RAG |
| G2 | Best T14 retrieval | 4 | Does better retrieval give better answers? (G1 → G2) |
| G3 | Best T14 retrieval | 8 | Does more context help, or does it confuse the small model? (G2 → G3) |

Answers are cached per experiment and split. **If you change the prompt or the evidence file, delete the matching `output/t15_*.json` cache files**, otherwise the old answers are loaded.

In [49]:
# T15.4: Run all generation experiments on the VALIDATION set

T15_CONFIGS = {
    "G0 closed-book (no evidence)":       lambda i: build_closed_prompt(i),
    "G1 M1 dense evidence, top-4":        lambda i: build_rag_prompt(i, "m1_dense", 4),
    "G2 best T14 evidence, top-4":        lambda i: build_rag_prompt(i, "best", 4),
    "G3 best T14 evidence, top-8":        lambda i: build_rag_prompt(i, "best", 8),
}

def cache_name(name, split):
    tag = name.split()[0]                                              # G0, G1, ...
    ev = os.path.splitext(os.path.basename(BEST_EVIDENCE_FILE))[0] if "best" in name else "fixed"
    return f"output/t15_{tag}_{ev}_{split}.json"

def run_config(name, split, batch_size=8):
    ids = splits[split]
    prompts = {i: T15_CONFIGS[name](i) for i in ids}
    n_tok = np.mean([len(tokenizer(prompts[i]).input_ids) for i in ids[:50]])
    t0 = time.perf_counter()
    preds = run_cached(prompts, cache_name(name, split), max_new_tokens=32,
                       batch_size=batch_size, desc=f"{name} [{split}]")
    sec = time.perf_counter() - t0
    timings[f"T15 {name} [{split}]"] = sec
    print(f"   avg prompt length ≈ {n_tok:.0f} tokens, {sec / len(ids):.2f} s/question "
          f"(time is only meaningful on the first, non-cached run)")
    return preds

val_preds = {name: run_config(name, "val") for name in T15_CONFIGS}

G0 closed-book (no evidence) [val]: 0 cached, 510 to generate


G0 closed-book (no evidence) [val]:   0%|          | 0/64 [00:00<?, ?it/s]

   avg prompt length ≈ 138 tokens, 0.09 s/question (time is only meaningful on the first, non-cached run)
G1 M1 dense evidence, top-4 [val]: 0 cached, 510 to generate


G1 M1 dense evidence, top-4 [val]:   0%|          | 0/64 [00:00<?, ?it/s]

   avg prompt length ≈ 1160 tokens, 0.29 s/question (time is only meaningful on the first, non-cached run)
G2 best T14 evidence, top-4 [val]: 0 cached, 510 to generate


G2 best T14 evidence, top-4 [val]:   0%|          | 0/64 [00:00<?, ?it/s]

   avg prompt length ≈ 1212 tokens, 0.30 s/question (time is only meaningful on the first, non-cached run)
G3 best T14 evidence, top-8 [val]: 0 cached, 510 to generate


G3 best T14 evidence, top-8 [val]:   0%|          | 0/64 [00:00<?, ?it/s]

   avg prompt length ≈ 2253 tokens, 0.53 s/question (time is only meaningful on the first, non-cached run)


### 15.5 – Compare on validation and choose the configuration

**Selection rule (fixed in advance):** among the RAG configurations G1–G3, we choose the one with the highest **accuracy over all validation questions**, including the null questions. This rewards correct answers and correct "I don't know"s equally. G0 is only a reference point.

In [50]:
# T15.5: Validation table and selection

val_answer_results = {name: answer_metrics(p, "val") for name, p in val_preds.items()}
print_answer_table(f"T15 – VALIDATION set ({len(splits['val'])} questions, incl. null)", val_answer_results)

rag_configs = [n for n in T15_CONFIGS if not n.startswith("G0")]
BEST_T15 = max(rag_configs, key=lambda n: val_answer_results[n]["Acc (all)"])
print(f"\nSelected on validation: {BEST_T15}")


T15 – VALIDATION set (510 questions, incl. null)
Experiment                                       All     Answ.    Infer.   Compar.   Tempor.     Null✓ FalseAbst
G0 closed-book (no evidence)                  0.2294    0.2467    0.3129    0.2865    0.0948    0.1000    0.4800
G1 M1 dense evidence, top-4                   0.4510    0.4200    0.7730    0.2573    0.1638    0.6833    0.4422
G2 best T14 evidence, top-4                   0.5039    0.4778    0.8589    0.3158    0.1810    0.7000    0.3533
G3 best T14 evidence, top-8                   0.5078    0.5044    0.8344    0.3567    0.2586    0.5333    0.3267

Selected on validation: G3 best T14 evidence, top-8



### 15.6 – Final evaluation on the test set (run once)

We run the closed-book reference (G0), the RAG baseline (G1) and the selected configuration on the test questions. On the L4 GPU used for this run, the three configurations took about 24 minutes in total (G0 2 min, G1 8 min, G3 14 min). On a T4 it takes roughly 2–3 times longer.

In [51]:
# T15.6: Test set – reference, baseline and the selected configuration

test_configs = ["G0 closed-book (no evidence)", "G1 M1 dense evidence, top-4"]
if BEST_T15 not in test_configs:
    test_configs.append(BEST_T15)

test_preds = {name: run_config(name, "test") for name in test_configs}
test_answer_results = {name: answer_metrics(p, "test") for name, p in test_preds.items()}
print_answer_table(f"T15 – TEST set ({len(splits['test'])} questions, incl. null)", test_answer_results)

G0 closed-book (no evidence) [test]: 0 cached, 1536 to generate


G0 closed-book (no evidence) [test]:   0%|          | 0/192 [00:00<?, ?it/s]

   avg prompt length ≈ 137 tokens, 0.09 s/question (time is only meaningful on the first, non-cached run)
G1 M1 dense evidence, top-4 [test]: 0 cached, 1536 to generate


G1 M1 dense evidence, top-4 [test]:   0%|          | 0/192 [00:00<?, ?it/s]

   avg prompt length ≈ 1146 tokens, 0.29 s/question (time is only meaningful on the first, non-cached run)
G3 best T14 evidence, top-8 [test]: 0 cached, 1536 to generate


G3 best T14 evidence, top-8 [test]:   0%|          | 0/192 [00:00<?, ?it/s]

   avg prompt length ≈ 2248 tokens, 0.54 s/question (time is only meaningful on the first, non-cached run)

T15 – TEST set (1536 questions, incl. null)
Experiment                                       All     Answ.    Infer.   Compar.   Tempor.     Null✓ FalseAbst
G0 closed-book (no evidence)                  0.2344    0.2487    0.4143    0.2140    0.0684    0.1271    0.5026
G1 M1 dense evidence, top-4                   0.4408    0.4059    0.7306    0.2354    0.2023    0.7017    0.4266
G3 best T14 evidence, top-8                   0.5026    0.4937    0.8265    0.3327    0.2650    0.5691    0.3358


### 15.7 – Look at the errors (validation set)

We read examples of the three typical failures on the **validation** set:
1. **Hallucination:** a null question where the model invented an answer instead of abstaining.
2. **False abstention:** an answerable question where the model said *Insufficient information*.
3. **Wrong answer:** the model answered, but incorrectly.

For each example we also print whether a gold evidence fact was among the chunks the model saw. This tells us whether the error came from **retrieval** (the evidence was missing) or from **generation** (the evidence was there, but the model misread it).

In [53]:
# T15.7: Error analysis on the validation set for the selected configuration

preds = val_preds[BEST_T15]
src = "m1_dense" if "M1" in BEST_T15 else "best"
k_used = 8 if "top-8" in BEST_T15 else 4

def evidence_seen(i):
    seen = " ".join(d["text"] for d in evidence[src][i]["retrieval_list"][:k_used])
    gold = [e["fact"] for e in questions[i]["evidence_list"]]
    return any(g in seen for g in gold) if gold else None

groups = {
    "Hallucination (null question, model answered)":
        [i for i in splits["val"] if questions[i]["question_type"] == "null_query" and not is_abstention(preds[i])],
    "False abstention (answerable, model gave up)":
        [i for i in splits["val"] if questions[i]["question_type"] != "null_query" and is_abstention(preds[i])],
    "Wrong answer":
        [i for i in splits["val"] if questions[i]["question_type"] != "null_query"
         and not is_abstention(preds[i]) and not is_correct(preds[i], questions[i]["answer"])],
}
for title, ids in groups.items():
    print(f"\n=== {title}: {len(ids)} cases ===")
    if ids and questions[ids[0]]["question_type"] != "null_query":
        share = np.mean([evidence_seen(i) for i in ids])
        print(f"Gold evidence was in the model's context for {share:.0%} of these cases")
    for i in ids[:3]:
        print(f"- Q: {questions[i]['query']}")
        print(f"  gold: {questions[i]['answer']!r} | predicted: {preds[i]!r} | evidence seen: {evidence_seen(i)}")


=== Hallucination (null question, model answered): 28 cases ===
- Q: Considering the environmental concerns highlighted by The Guardian and the economic impacts discussed by The Wall Street Journal, which single letter represents both the primary Great Lake at risk due to invasive carp species and the Great Lake bordering the state that could face significant financial consequences from disrupted fisheries?
  gold: 'Insufficient information.' | predicted: 'MI' | evidence seen: None
- Q: Considering the environmental initiatives discussed in a Bloomberg article and the new seasonal drink flavors introduced according to a CNN report, which letter represents both the first character of the initiative aimed at reducing waste at Starbucks and the initial of the primary spice used in Starbucks' latest autumn-themed beverage?
  gold: 'Insufficient information.' | predicted: 'S' | evidence seen: None
- Q: Considering the information from a Bloomberg article and a report by The Wall Street Jou

### 15.8 – Save predictions, results and timings

The test predictions of the selected configuration are saved with the question, gold answer and question type, so Milestone 3 (unanswerable detection, LLM-as-judge vs. manual checks) can start directly from this file.

In [54]:
# T15.8: Save everything needed for the README and Milestone 3

final = [{"query": questions[i]["query"], "question_type": questions[i]["question_type"],
          "gold_answer": questions[i]["answer"], "prediction": test_preds[BEST_T15][i],
          "correct": bool(is_correct(test_preds[BEST_T15][i], questions[i]["answer"]))}
         for i in splits["test"]]
with open("output/t15_final_predictions_test.json", "w", encoding="utf-8") as f:
    json.dump(final, f, ensure_ascii=False, indent=2)

with open("output/t15_results.json", "w") as f:
    json.dump({"validation": val_answer_results, "test": test_answer_results,
               "selected": BEST_T15, "best_evidence_file": BEST_EVIDENCE_FILE,
               "baseline_evidence_file": BASELINE_EVIDENCE_FILE, "llm": LLM_NAME,
               "quantization": "4-bit NF4 (bitsandbytes), double quant, fp16 compute",
               "decoding": "greedy, max_new_tokens=32", "timings_seconds": timings}, f, indent=2)

print("Saved: output/t15_final_predictions_test.json")
print("Saved: output/t15_results.json")
print("\nTime per stage:")
for k, v in timings.items():
    print(f"  {k:60s} {v:8.1f} s")

Saved: output/t15_final_predictions_test.json
Saved: output/t15_results.json

Time per stage:
  T14 build retrievers                                             69.5 s
  T14 dense retrieval, original (2556 q)                            4.2 s
  T14 BM25 retrieval, original (2556 q)                             1.4 s
  T14 load 4-bit LLM                                               29.9 s
  T14 decomposition (LLM, 2556 q)                                1156.5 s
  T14 dense+BM25 retrieval, sub-questions (5939 q)                  6.6 s
  T14 rerank original pool (51120 pairs)                          403.3 s
  T14 rerank decomposed pool (51120 pairs)                        401.6 s
  T15 G0 closed-book (no evidence) [val]                           44.6 s
  T15 G1 M1 dense evidence, top-4 [val]                           149.6 s
  T15 G2 best T14 evidence, top-4 [val]                           150.5 s
  T15 G3 best T14 evidence, top-8 [val]                           268.7 s
  T15 G0 closed-bo

## T15 – Result

**Validation set** (510 questions, including 60 null questions)

| ID | Evidence | k | Acc (all) | Acc (answerable) | Null abstained | Wrongly abstained |
|---|---|---|---|---|---|---|
| G0 | none (closed-book) | 0 | 0.2294 | 0.2467 | 0.1000 | 0.4800 |
| G1 | M1 dense (baseline) | 4 | 0.4510 | 0.4200 | 0.6833 | 0.4422 |
| G2 | best T14 (R6) | 4 | 0.5039 | 0.4778 | **0.7000** | 0.3533 |
| G3 | best T14 (R6) | 8 | **0.5078** | **0.5044** | 0.5333 | **0.3267** |

**Selected on validation:** G3 (highest accuracy on all questions, the rule fixed in 15.5).

**Test set** (1536 questions, including 181 null questions)

| ID | Evidence | k | Acc (all) | Acc (answerable) | Inference | Comparison | Temporal | Null abstained | Wrongly abstained |
|---|---|---|---|---|---|---|---|---|---|
| G0 | none (closed-book) | 0 | 0.2344 | 0.2487 | 0.4143 | 0.2140 | 0.0684 | 0.1271 | 0.5026 |
| G1 | M1 dense (baseline) | 4 | 0.4408 | 0.4059 | 0.7306 | 0.2354 | 0.2023 | **0.7017** | 0.4266 |
| G3 | best T14 (R6) | 8 | **0.5026** | **0.4937** | **0.8265** | **0.3327** | **0.2650** | 0.5691 | **0.3358** |

**Discussion**
- **G0 → G1:** retrieval almost doubles the accuracy (test 0.23 → 0.44). The 3B model does not know these 2023 news articles, so it needs the evidence.
- **G1 → G2/G3:** better retrieval (T12–T14) turns into better answers. On test, the selected G3 is **+6.2 points** better than the baseline G1 on all questions, and +8.8 points on answerable questions.
- **G2 vs. G3:** G3 won by only 0.4 points (2 of 510 questions). With 8 chunks the model answers more answerable questions correctly, but it **abstains less on null questions** (validation 0.70 → 0.53, test G1 0.70 → G3 0.57). More context makes the model more confident, including when it should say "I don't know". G3 also takes about twice as long per question (≈0.54 s vs ≈0.30 s on L4). In our first (interrupted) run on a T4, G2 was ahead of G3 instead, so the difference between them is within run-to-run noise.
- **Hardest types:** temporal (0.27) and comparison (0.33) questions. They need two facts plus a comparison of dates or claims, which is hard for a 3B model. Inference questions are easy once the evidence is found (0.83).
- **Errors (15.7, validation, G3):** 28 of 60 null questions were answered instead of abstained (hallucinations). There were 147 false abstentions, and 76 wrong answers. In 89% of the false abstentions and 96% of the wrong answers, the gold evidence **was** in the model's context. So most remaining errors come from **generation**, not from retrieval.
- **Main takeaway for Milestone 3:** the small model trades hallucination against false abstention. Improving how it decides to say "Insufficient information." is the clearest next step.